![](https://github.com/ibmm-unibe-ch/FrankenMSA/blob/dev/app/assets/frankenmsa_header.png?raw=true)

# FrankenMSA-Colab
This notebook launches the [FrankenMSA App](https://github.com/ibmm-unibe-ch/FrankenMSA/tree/main/) **in Google Colab** to provide a GUI for manipulating Multiple Sequence Alignments (MSAs).


Tip: use “Runtime” → “Run all” (or `Ctrl + F9`) to execute all cells.

In [ ]:
FRANKEN_GIT_BRANCH = "dev-pilot" # change as needed (default: main)
FRANKEN_GIT_URL = "https://github.com/ibmm-unibe-ch/FrankenMSA.git"

In [ ]:
#@title Install Prerequisites

import os
os.system("pip install termcolor gitpython ipywidgets ipython python-dotenv > /dev/null 2>&1")

import git, sys, importlib
from pathlib import Path
from termcolor import colored
import dotenv

dotenv.load_dotenv()
os.environ["FRANKEN_RUNTIME"] = "colab"


def warn(msg):
    print(colored("[WARNING] ", "yellow") + msg, file=sys.stderr)

def info(msg):
    print(colored("[INFO] ", "cyan") + msg)

info("Installed prerequisite packages.")

In [ ]:
#@markdown ### Installation settings
install_ProteinMPNN = False #@param {type:"boolean"}
install_GhostFold = False #@param {type:"boolean"}
install_ESM3 = False #@param {type:"boolean"}

#Only dummy right now, but would be nice to have?

In [ ]:
#@title Forwarding Options

#@markdown Choose how to forward the app from Colab to your browser. Either use Colab's native port forwarding or ngrok (requires auth token).

FORWARDING_METHOD = "native" #@param ["native", "ngrok"]
PORT = 8050 #@param {type:"integer"}

use_ngrok = lambda: FORWARDING_METHOD == "ngrok"

if FORWARDING_METHOD == "ngrok":
    print("ℹ️ ngrok forwarding will create a shareable public link (requires auth token)")
else:
    print("ℹ️ Google Native forwarding uses Colab's built-in port forwarding")


In [ ]:
#@title Install FrankenMSA
import importlib.util
FORCE_REINSTALL = False #@param {type:"boolean"}
if install_GhostFold and (FORCE_REINSTALL or (not Path("ghostfold").exists())):
    !pip install biopython numpy matplotlib scikit-learn
    !git clone https://github.com/rostro36/ghostfold.git
    %cd ghostfold
    !chmod +x ghostfold.sh
    !hf auth login
    %cd ..

if install_ESM3 and (FORCE_REINSTALL or (importlib.util.find_spec("esm") is None)):
    print("installing ESM")
    %pip install esm
    from esm.models.esmc import ESMC
    client = ESMC.from_pretrained("esmc_300m")

if Path("frankenmsa").exists() and not FORCE_REINSTALL:
    warn("FrankenMSA directory already exists; skipping clone.")
else:
    os.system("rm -rf FrankenMSA frankenmsa app; rm -f *.py")
    info(f"Cloning FrankenMSA from {FRANKEN_GIT_URL} (branch: {FRANKEN_GIT_BRANCH})...")
    git.Repo.clone_from(FRANKEN_GIT_URL, "FrankenMSA", branch=FRANKEN_GIT_BRANCH)
    info("FrankenMSA cloned.")

    os.system("mv FrankenMSA/app .; mv FrankenMSA/frankenmsa .; mv FrankenMSA/setup.py .; rm -rf FrankenMSA; pip install -e .")
    info("FrankenMSA installed.")

# kill any existing instances
!pkill -f "app/app.py" 2>/dev/null || true
!pkill -f "gunicorn" 2>/dev/null || true
!pkill -f "ngrok" 2>/dev/null || true

In [ ]:
#@title Launch FrankenMSA App

import subprocess
import time
import threading
import getpass
import re
from google.colab import output as colab_output

from frankenmsa.runtime import build_app_launch_env
from frankenmsa.runtime import normalize_runtime_environment

normalize_runtime_environment(runtime="colab")

def launch_app_subprocess(port, host, render_mode, use_ngrok, public_url=None):
    """Launch FrankenMSA app as subprocess and wait for startup."""
    env = build_app_launch_env(
        port=port,
        host=host,
        render_mode=render_mode,
        runtime="colab",
        use_ngrok=use_ngrok,
        public_url=public_url,
        base_env=os.environ,
        pythonpath_prefix="/content",
    )
    
    proc = subprocess.Popen(
        [sys.executable, "app/app.py"],
        cwd=str(Path.cwd().resolve()),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    
    start = time.time()
    lines = []
    app_started = False
    
    while time.time() - start < 25:
        ln = proc.stdout.readline()
        if ln:
            lines.append(ln.rstrip())
            if "Dash is running" in ln or "Running on" in ln:
                app_started = True
                break
        else:
            time.sleep(0.2)
    
    return proc, app_started, lines


def tail_logs(proc):
    """Tail subprocess logs in background thread."""
    def _tail():
        while True:
            line = proc.stdout.readline()
            if not line:
                time.sleep(0.2)
                continue
            print(line, end="")
    
    thread = threading.Thread(target=_tail, daemon=True)
    thread.start()
    return thread


def get_or_create_ngrok_tunnel(port):
    """Get existing ngrok tunnel or create new one."""
    try:
        import pyngrok
    except:
        info("Installing pyngrok...")
        os.system("pip install pyngrok > /dev/null 2>&1")
    
    from pyngrok import ngrok, conf
    
    token = os.environ.get("NGROK_AUTH_TOKEN", "").strip()
    if not token:
        warn("No ngrok auth token found in NGROK_AUTH_TOKEN env variable.")
        warn("You can sign up for a free ngrok account at https://ngrok.com/")
        token = getpass.getpass("Enter ngrok authtoken (hidden): ").strip().strip("'").strip('"')
        os.environ["NGROK_AUTH_TOKEN"] = token
        info("ngrok auth token set as environment variable.")
    
    conf.get_default().auth_token = token
    
    public_url = None
    try:
        for tunnel in ngrok.get_tunnels():
            addr = (tunnel.config or {}).get("addr", "")
            if addr.endswith(f":{port}"):
                public_url = tunnel.public_url
                print(f"♻️ Reusing existing tunnel: {public_url}")
                return public_url
        
        tunnel = ngrok.connect(addr=f"0.0.0.0:{port}", proto="http")
        public_url = tunnel.public_url
        print(f"✅ Created new tunnel: {public_url}")
        return public_url
    
    except Exception as e:
        msg = str(e)
        match = re.search(r"https?://[a-z0-9\-]+\.ngrok-[\w\-]+\.(?:dev|app)", msg)
        if match:
            public_url = match.group(0)
            warn(f"♻️ Using tunnel from error message: {public_url}")
            return public_url
        raise e


# === MAIN LOGIC ===

if not use_ngrok():
    info(f"🌐 Launching FrankenMSA on port {PORT} with Colab native forwarding...")
    
    proc, app_started, startup_lines = launch_app_subprocess(
        PORT, "0.0.0.0", "inline", False
    )
    
    if app_started:
        colab_output.serve_kernel_port_as_window(PORT)
        info("✅ A link to the app should now be displayed above")
    else:
        warn("App may not have started properly. Check logs below.")
        print("\n---- recent logs ----")
        print("\n".join(startup_lines))
        print("---------------------")
    
    tail_logs(proc)
    info("📡 App logs are being tailed in the background")

else:
    info("🌐 Setting up ngrok tunnel...")
    public_url = get_or_create_ngrok_tunnel(PORT)
    
    info(f"🌐 Launching FrankenMSA on port {PORT} via ngrok tunnel...")
    
    proc, app_started, startup_lines = launch_app_subprocess(
        PORT, "0.0.0.0", "external", True, public_url=public_url
    )
    
    print("\n---- startup logs ----")
    print("\n".join(startup_lines[-20:]))
    print("---------------------")
    info(f"🌐 Open: {public_url}")
    print("📡 Tailing FrankenMSA app logs (Ctrl+C to stop):")
    
    tail_logs(proc)